# 사용자실험: Pearson 상관분석

사용자실험에서 수집한 7개 설문 종속변수 사이의 선형 관계를 Pearson 상관계수로 분석합니다. 마지막에는 평균, 표준편차와 상관계수를 함께 제시하는 논문용 상관표를 출력합니다.

> 인지부하는 점수가 높을수록 부하가 크므로 긍정 방향 변수와 음의 상관이 나타날 수 있습니다.


## 분석 순서

1. 설문 원자료와 분석 문항을 준비합니다.
2. 참가자별 종속변수 점수를 계산합니다.
3. 기술통계와 Pearson 상관계수를 계산합니다.
4. 각 상관계수의 양측 p값을 계산합니다.
5. 별표와 변수 번호를 적용한 논문용 표를 만듭니다.


## 1. 데이터 준비

### 1-1. 라이브러리와 원자료 불러오기

`scipy.stats`의 `pearsonr()`를 사용해 상관계수와 양측 p값을 계산합니다. 참가자 개인정보는 출력하지 않고 자료의 크기만 확인합니다.


In [ ]:
from pathlib import Path
import sys

A2UI_DIR = Path.cwd() if (Path.cwd() / "analysis_common.py").exists() else Path.cwd() / "Data" / "A2UI"
if str(A2UI_DIR) not in sys.path:
    sys.path.insert(0, str(A2UI_DIR))
import analysis_common as common
from functools import partial
from analysis_common import significance_stars, format_p

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

survey = common.load_survey()

print(f"자료 크기: {survey.shape[0]}명 × {survey.shape[1]}개 열")


### 1-2. 종속변수별 문항 정의

다른 사용자실험 분석 노트북과 동일한 변수명과 문항 구성을 사용합니다.


In [ ]:
DV_ITEMS = common.DV_ITEMS

item_map = pd.DataFrame({
    "종속변수": list(DV_ITEMS),
    "사용 문항": [", ".join(items) for items in DV_ITEMS.values()],
})
display(item_map)


### 1-3. 문항 열 찾기와 평균 계산

문항 번호로 실제 엑셀 열을 찾은 뒤 참가자별 문항 평균을 계산합니다. 결측값이나 숫자가 아닌 응답이 있으면 실행을 중단합니다.


In [ ]:
find_columns = partial(common.find_columns, survey)
average_score = partial(common.average_score, survey)


### 1-4. 참가자별 7개 종속변수 점수

구성개념 $c$의 참가자별 점수는 해당 문항들의 평균입니다.

$$Y_{ic}=\frac{1}{k_c}\sum_{j=1}^{k_c}Q_{icj}$$

재사용 의도는 단일 문항이므로 문항 값 자체가 점수가 됩니다.


In [ ]:
scores = common.make_scores(survey)

assert scores.shape == (40, 7), "종속변수 점수표의 크기가 예상과 다릅니다."
assert scores.notna().all().all(), "종속변수 점수에 결측값이 있습니다."

display(scores.head().round(2))


---

## 2. 기술통계 계산

논문용 상관표에 포함할 전체 표본의 평균(`M`)과 표본표준편차(`SD`)를 계산합니다.


In [ ]:
descriptive_statistics = pd.DataFrame({
    "M": scores.mean(),
    "SD": scores.std(ddof=1),
})
display(descriptive_statistics.round(2))


>7점 척도가 1~7점이므로 중앙값 성격의 기준점은 4점이다. 
다라서 예를 들어 PU가 5.56이라면, 참가자들이 전반적으로 해당 인터페이스가 과제 수행에 유용했다고 비교적 긍정적으로 판단.

## 3. Pearson 상관계수와 p값

### 3-1. Pearson 상관계수

$$r=\frac{\sum_{i=1}^{n}(x_i-\bar{x})(y_i-\bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i-\bar{x})^2\sum_{i=1}^{n}(y_i-\bar{y})^2}}$$

$r$은 -1에서 1 사이이며 절댓값이 클수록 선형 관계가 강합니다. `scores.corr()`로 전체 상관계수 행렬을 계산합니다.


In [ ]:
variables = list(scores.columns)
N = len(scores)
CORRELATION_DF = N - 2
r_matrix = pd.DataFrame(np.eye(len(variables)), index=variables, columns=variables)
p_matrix = pd.DataFrame(np.nan, index=variables, columns=variables)
for i, left in enumerate(variables):
    for right in variables[i + 1:]:
        result = stats.pearsonr(scores[left], scores[right])
        r_matrix.loc[left, right] = r_matrix.loc[right, left] = result.statistic
        p_matrix.loc[left, right] = p_matrix.loc[right, left] = result.pvalue
display(r_matrix.round(3))


r = .920 상당히 강한 양의 관계 
즉, 해당 인터페이스 유용하다고 평가한 참가자일수록 전반적인 만족도 역시 높게 평가하는 경향이 매우 뚜렸하였다. 

but, 유용성이 만족도를 증가시켰다라고 쓰면 안된다. 산관분석은 관계를 보여주는 것이지 인관관계를 증명하는 분석이 아니기 때문이다. 

### 3-2. 양측 유의성 검정

> 양측 유의성 검정을 왜 하는가? 
양측 유의성 검정 (Two-tailed test)을 수행하는 핵심 이유는 연구자가 예상한 방향뿐만 아니라 그 반대 방향의 결과까지 모두 고려하여, 변수 간의 관계나 차이가 존재하는지를 가장 객관적이고 보수적으로 검증하기 위해서이다. 

계산과정 : 참가자가 40명이므로 각 상관계수 검정의 자유도는 $n-2=38$입니다. 변수 쌍마다 `pearsonr()`를 실행해 양측 p값을 구합니다. 대각선은 동일한 변수의 상관이므로 계산하지 않습니다.


In [ ]:
display(p_matrix.round(4))


### 3-3. 유의성 별표 표시

논문 표에 사용할 별표 기준은 공통 모듈의 함수를 사용합니다.

- `**`: $p<.05$
- `*`: $.05<p<.10$

p=.05와 p=.10은 별표 없이 표시하며, 반올림 전 p값으로 판정합니다.


## 4. 논문용 기술통계 및 상관표

중복을 피하기 위해 상관계수는 하삼각 영역만 표시합니다. 행에는 번호와 변수명을 함께 쓰고, 열에는 `M`, `SD`와 변수 번호를 배치합니다. 각 상관계수는 소수 둘째 자리와 유의성 별표로 표시합니다.


In [ ]:
variable_numbers = {variable: str(index + 1) for index, variable in enumerate(variables)}
row_labels = [f"{variable_numbers[variable]}. {variable}" for variable in variables]
correlation_columns = ["M", "SD"] + list(variable_numbers.values())

publication_correlation_table = pd.DataFrame(
    "",
    index=row_labels,
    columns=correlation_columns,
)

for row_index, variable in enumerate(variables):
    row_label = row_labels[row_index]
    publication_correlation_table.loc[row_label, "M"] = f"{scores[variable].mean():.2f}"
    publication_correlation_table.loc[row_label, "SD"] = f"{scores[variable].std(ddof=1):.2f}"
    publication_correlation_table.loc[row_label, str(row_index + 1)] = "—"

    for column_index in range(row_index):
        other_variable = variables[column_index]
        r_value = r_matrix.loc[variable, other_variable]
        p_value = p_matrix.loc[variable, other_variable]
        stars = significance_stars(p_value)
        publication_correlation_table.loc[row_label, str(column_index + 1)] = f"{r_value:.2f}{stars}"

publication_correlation_table.index.name = "변수"
display(publication_correlation_table)
print("주. N=40. 값은 Pearson의 r이며 양측검정을 사용하였다. ** p<.05, * .05<p<.10. p=.05와 p=.10은 별표 없음.")


인지부하 <-> 결과신뢰도, 품질 r = -.294 이고 P = 0.065 따라서 P>.05 통계적으로 유의하지 않다.
> 인지부하가 낮을수록 결과 신뢰도, 품질이 높아지는 방향성은 관찰 되었지만, 그 관계는 통계적으로 유의하지 않았다. 

---

## 5. 논문용 시각화: 상관행렬 히트맵

변수 간 관계의 방향과 크기를 빠르게 파악할 수 있도록 하삼각 상관행렬을 시각화합니다. 같은 정보를 중복하지 않도록 대각선과 상삼각 영역은 표시하지 않습니다.


In [ ]:
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

KOREAN_FONT = common.configure_plots()
FIGURE_DIR = common.FIGURE_DIR


**그림 3. 사용자실험 설문 종속변수 간 Pearson 상관계수.** PU=지각된 유용성, PEU=사용 용이성, SAT=전반적 만족도, CL=인지부하, TQ=결과 신뢰도·품질, IQ=상호작용 품질, RI=재사용 의도. 색상과 셀 안의 숫자는 상관의 방향과 크기를 나타낸다.


In [ ]:
short_labels = ["PU", "PEU", "SAT", "CL", "TQ", "IQ", "RI"]
heatmap_data = r_matrix.copy()
heatmap_data.index = short_labels
heatmap_data.columns = short_labels
upper_triangle_mask = np.triu(np.ones_like(heatmap_data, dtype=bool), k=0)

heatmap_annotations = np.array([[f"{r_matrix.iloc[i, j]:.2f}{significance_stars(p_matrix.iloc[i, j])}" for j in range(len(variables))] for i in range(len(variables))])

fig, ax = plt.subplots(figsize=(7.0, 5.8))
sns.heatmap(
    heatmap_data,
    mask=upper_triangle_mask,
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    annot=heatmap_annotations,
    fmt="",
    linewidths=.7,
    linecolor="white",
    square=True,
    cbar_kws={"label": "Pearson r", "shrink": .78},
    ax=ax,
)
ax.set_title("사용자실험 설문 종속변수 간 상관", pad=12)
ax.set_xlabel("변수")
ax.set_ylabel("변수")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)
fig.text(.5, .01, "** p<.05; * .05<p<.10", ha="center", fontsize=9)
fig.tight_layout(rect=[0, .035, 1, 1])

fig.savefig(FIGURE_DIR / "figure_03_correlation_heatmap.png", dpi=300, bbox_inches="tight")
fig.savefig(FIGURE_DIR / "figure_03_correlation_heatmap.pdf", bbox_inches="tight")
plt.show()
